# Run All

Runs the whole pipeline in order, from the raw CSV files to the deep-dive analysis:

| Step | Notebook |
| --- | --- |
| 1 | `01_data_setup.ipynb` |
| 2 | `02_data_quality.ipynb` |
| 3 | `03_data_cleaning.ipynb` |
| 4 | `04_delay_analysis.ipynb` |
| 5 | `05_deep_dive_analysis.ipynb` |

Each notebook is executed in a fresh kernel and saved in place with its outputs. The run stops at the first notebook that fails.

## 1. Setup

In [1]:
import time
from pathlib import Path

import nbformat
from nbconvert.preprocessors import CellExecutionError, ExecutePreprocessor

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

PIPELINE = [
    "01_data_setup.ipynb",
    "02_data_quality.ipynb",
    "03_data_cleaning.ipynb",
    "04_delay_analysis.ipynb",
    "05_deep_dive_analysis.ipynb",
]
TIMEOUT_SEC = 1200

missing = [name for name in PIPELINE if not (NOTEBOOK_DIR / name).is_file()]
assert not missing, f"Missing notebooks: {missing}"
print(f"Project root: {PROJECT_ROOT}")

Project root: e:\Important\Skills\railpulse-railway-analytics


## 2. Run the notebooks

The working directory is the notebooks folder, the same as when each notebook is opened by hand.

In [2]:
results = []

for name in PIPELINE:
    path = NOTEBOOK_DIR / name
    nb = nbformat.read(path, as_version=4)
    executor = ExecutePreprocessor(timeout=TIMEOUT_SEC, kernel_name="python3")

    start = time.time()
    try:
        executor.preprocess(nb, {"metadata": {"path": str(NOTEBOOK_DIR)}})
        status = "OK"
    except CellExecutionError as err:
        status = "FAILED"
        error = err
    finally:
        # keep the outputs, including those of a failed run, for debugging
        nbformat.write(nb, path)

    results.append((name, status, time.time() - start))
    print(f"{status:7} {name} ({results[-1][2]:.1f}s)")
    if status == "FAILED":
        raise error

e:\Important\Skills\railpulse-railway-analytics\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()


OK      01_data_setup.ipynb (1.4s)
OK      02_data_quality.ipynb (1.6s)
OK      03_data_cleaning.ipynb (1.3s)
OK      04_delay_analysis.ipynb (3.4s)
OK      05_deep_dive_analysis.ipynb (14.3s)


## 3. Summary

In [3]:
import pandas as pd

summary = pd.DataFrame(results, columns=["notebook", "status", "seconds"]).round({"seconds": 1})
summary

,notebook,status,seconds
0,01_data_setup.ipynb,OK,1.4
1,02_data_quality.ipynb,OK,1.6
2,03_data_cleaning.ipynb,OK,1.3
3,04_delay_analysis.ipynb,OK,3.4
4,05_deep_dive_analysis.ipynb,OK,14.3
